In [ ]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

# Load the combined ego-Facebook edge list directly from the SNAP URL
data_url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
edges = pd.read_csv(data_url, sep=" ", names=["source", "target"], compression="gzip")

all_nodes = pd.concat([edges["source"], edges["target"]])
degree_counts = all_nodes.value_counts().reset_index()
degree_counts.columns = ["node_id", "degree"]

ego_ids = [0, 107, 348, 414, 686, 698, 1684, 1912, 3437, 3980]
degree_counts["node_type"] = np.where(
    degree_counts["node_id"].isin(ego_ids), "Ego Center (Surveyed)", "Alter (Friend)"
)

ego_edges = edges[edges["source"].isin(ego_ids) | edges["target"].isin(ego_ids)].copy()
ego_edges["ego_network"] = np.where(
    ego_edges["source"].isin(ego_ids), ego_edges["source"], ego_edges["target"]
)

ego_edges["member_node"] = np.where(
    ego_edges["source"].isin(ego_ids), ego_edges["target"], ego_edges["source"]
)

node_to_ego = (
    ego_edges.groupby("member_node")["ego_network"]
    .min()
    .reset_index()
    .rename(columns={"member_node": "node_id"})
)

ego_self = pd.DataFrame({"node_id": ego_ids, "ego_network": ego_ids})
node_to_ego = pd.concat([ego_self, node_to_ego]).drop_duplicates(subset=["node_id"])

nodes_df = degree_counts.merge(node_to_ego, on="node_id", how="left")
nodes_df["ego_network"] = nodes_df["ego_network"].fillna(-1).astype(int)
nodes_df["ego_label"] = "Ego " + nodes_df["ego_network"].astype(str)


# 1. Degree Distribution Across the Facebook Network

ego_options = [None] + sorted(nodes_df["ego_label"].unique().tolist())
ego_labels = ["All Ego Networks"] + sorted(nodes_df["ego_label"].unique().tolist())

ego_dropdown = alt.binding_select(
    options=ego_options, labels=ego_labels, name="Filter by Ego Subnetwork: "
)
ego_select = alt.selection_point(fields=["ego_label"], bind=ego_dropdown)

plot1 = (
    alt.Chart(nodes_df)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X(
            "degree:Q",
            bin=alt.Bin(maxbins=35),
            title="Node Degree (Number of Facebook Friends)",
        ),
        y=alt.Y(
            "count():Q",
            scale=alt.Scale(type="symlog"),
            title="Number of Users (Symmetric Log Scale)",
        ),
        color=alt.Color(
            "node_type:N",
            scale=alt.Scale(
                domain=["Alter (Friend)", "Ego Center (Surveyed)"],
                range=["#4c78a8", "#e45756"],
            ),
            title="User Role in Network",
        ),
        tooltip=[
            alt.Tooltip("node_type:N", title="Role"),
            alt.Tooltip("degree:Q", bin=alt.Bin(maxbins=35), title="Degree Range"),
            alt.Tooltip("count():Q", title="User Count"),
        ],
    )
    .add_params(ego_select)
    .transform_filter(ego_select)
    .properties(
        width=600,
        height=350,
        title="Distribution of Friend Counts (Node Degree) in the SNAP ego-Facebook Network",
    )
)

# 2. Ego Subnetwork Size vs. Average Connectivity
ego_summary = (
    nodes_df[nodes_df["ego_network"].isin(ego_ids)]
    .groupby(["ego_network", "ego_label"], as_index=False)
    .agg(
        total_members=("node_id", "count"),
        mean_degree=("degree", "mean"),
        max_degree=("degree", "max"),
    )
)
ego_summary["mean_degree"] = ego_summary["mean_degree"].round(1)

bar_select = alt.selection_point(fields=["ego_label"], on="click", empty=True)

plot2 = (
    alt.Chart(ego_summary)
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    .encode(
        x=alt.X(
            "ego_label:N",
            sort="-y",
            title="Ego Subnetwork (Surveyed Core User ID)",
        ),
        y=alt.Y(
            "total_members:Q",
            title="Primary Subnetwork Size (Number of Users)",
        ),
        color=alt.Color(
            "mean_degree:Q",
            scale=alt.Scale(scheme="viridis"),
            title="Mean Friend Degree",
        ),
        opacity=alt.condition(bar_select, alt.value(1.0), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("ego_label:N", title="Ego Network"),
            alt.Tooltip("total_members:Q", title="Subnetwork Users"),
            alt.Tooltip("mean_degree:Q", title="Avg Friends per User"),
            alt.Tooltip("max_degree:Q", title="Max Friends in Subnetwork"),
        ],
    )
    .add_params(bar_select)
    .properties(
        width=600,
        height=350,
        title="Ego Subnetwork Size Colored by Average Friend Connectivity (Click a Bar to Highlight)",
    )
)

plot1.save("../assets/json/hw5_facebook_plot1.json")
plot2.save("../assets/json/hw5_facebook_plot2.json")
print("Saved both JSON files successfully!")